In [15]:
"""
ingestion.py — Canonical record extraction, row hashing, and change detection.
"""
import hashlib
import json
import logging
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple

import sys
import os

backend_path = os.path.abspath("../../../../")
sys.path.insert(0, backend_path)

# print(backend_path)
from backend.src.config import DATA_DIR, logger
from backend.src.schema import SCHEMA_CONFIG, load_csv


In [7]:
@dataclass
class CanonicalRecord:
    """Canonical representation of an entity record shared by graph and wiki."""
    entity_type: str
    entity_id: str
    display_name: str
    properties: Dict[str, Any]
    foreign_keys: Dict[str, Tuple[str, str, str]]  # fk_col -> (target_label, target_pk, target_val)
    source_file: str
    row_hash: str


In [8]:

def compute_row_hash(row: Dict[str, str]) -> str:
    """Deterministic hash of a single CSV row for change detection."""
    sorted_items = sorted(f"{k}:{v.strip()}" for k, v in row.items())
    raw = "|".join(sorted_items).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()[:16]

In [9]:
def parse_cell_value(val: str) -> Any:
    """Parse JSON strings (like JSON arrays) or return stripped string."""
    if not val:
        return ""
    val_clean = val.strip()
    if (val_clean.startswith("[") and val_clean.endswith("]")) or (val_clean.startswith("{") and val_clean.endswith("}")):
        try:
            return json.loads(val_clean)
        except Exception:
            return val_clean
    return val_clean

In [10]:
def extract_canonical_records() -> Dict[str, List[CanonicalRecord]]:
    """Extract canonical entity records from all configured CSV files."""
    records_by_type: Dict[str, List[CanonicalRecord]] = {}

    for filename, config in SCHEMA_CONFIG.items():
        label = config["label"]
        pk_col = config["pk"]
        display_col = config.get("display_name", pk_col)
        fk_map = config.get("fk", {})
        prop_cols = config.get("properties", [])

        rows = load_csv(filename)
        type_records: List[CanonicalRecord] = []

        for row in rows:
            entity_id = row.get(pk_col, "").strip()
            if not entity_id:
                continue

            properties: Dict[str, Any] = {}
            for col in prop_cols:
                properties[col] = parse_cell_value(row.get(col, ""))

            fks: Dict[str, Tuple[str, str, str]] = {}
            for fk_col, (target_label, target_pk) in fk_map.items():
                val = row.get(fk_col, "").strip()
                if val:
                    fks[fk_col] = (target_label, target_pk, val)

            display_name = str(row.get(display_col, entity_id)).strip()
            if len(display_name) > 120:
                display_name = display_name[:117] + "..."

            rec = CanonicalRecord(
                entity_type=label,
                entity_id=entity_id,
                display_name=display_name,
                properties=properties,
                foreign_keys=fks,
                source_file=filename,
                row_hash=compute_row_hash(row),
            )
            type_records.append(rec)

        records_by_type[label] = type_records

    return records_by_type


In [11]:
def detect_changes(
    current_records: Dict[str, List[CanonicalRecord]],
    previous_hashes: Optional[Dict[str, str]] = None
) -> Dict[str, Any]:
    """Detect added, updated, and deleted entities compared to previous manifest."""
    prev = previous_hashes or {}
    curr_hashes: Dict[str, str] = {}
    
    for label, records in current_records.items():
        for rec in records:
            curr_hashes[rec.entity_id] = rec.row_hash

    added = [eid for eid in curr_hashes if eid not in prev]
    updated = [eid for eid in curr_hashes if eid in prev and prev[eid] != curr_hashes[eid]]
    deleted = [eid for eid in prev if eid not in curr_hashes]

    return {
        "added": added,
        "updated": updated,
        "deleted": deleted,
        "is_unchanged": len(added) == 0 and len(updated) == 0 and len(deleted) == 0,
        "current_hashes": curr_hashes,
    }


In [13]:
import csv
def load_csv(filename: str) -> List[Dict[str, str]]:
    """Load a CSV file from DATA_DIR as list of dicts."""
    path = DATA_DIR / filename
    if not path.exists():
        logger.warning(f"File not found: {path}")
        return []
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return list(reader)


In [17]:
import sys
import os

backend_path = os.path.abspath("../../../../")
sys.path.insert(0, backend_path)

print(backend_path)

c:\Users\ps302\OneDrive\Desktop\garph hacks


In [22]:
load_csv(r"departments.csv")

[{'department_id': 'DEPT-ENG',
  'department_name': 'Engineering',
  'mission': 'Build and operate reliable product software',
  'head_employee_id': 'EMP-0001',
  'created_at': '2025-01-01T09:00:00Z'},
 {'department_id': 'DEPT-PROD',
  'department_name': 'Product',
  'mission': 'Define customer problems, roadmap, and product outcomes',
  'head_employee_id': 'EMP-0002',
  'created_at': '2025-01-01T09:00:00Z'},
 {'department_id': 'DEPT-DATA',
  'department_name': 'Data & AI',
  'mission': 'Build analytics, ML, and AI-powered capabilities',
  'head_employee_id': 'EMP-0003',
  'created_at': '2025-01-01T09:00:00Z'},
 {'department_id': 'DEPT-DESIGN',
  'department_name': 'Design & Research',
  'mission': 'Create accessible, usable product experiences',
  'head_employee_id': 'EMP-0004',
  'created_at': '2025-01-01T09:00:00Z'},
 {'department_id': 'DEPT-SALES',
  'department_name': 'Sales',
  'mission': 'Acquire customers and manage the sales pipeline',
  'head_employee_id': 'EMP-0005',
  'crea

In [23]:
def extract_canonical_records() -> Dict[str, List[CanonicalRecord]]:
    """Extract canonical entity records from all configured CSV files."""
    records_by_type: Dict[str, List[CanonicalRecord]] = {}

    for filename, config in SCHEMA_CONFIG.items():
        label = config["label"]
        pk_col = config["pk"]
        display_col = config.get("display_name", pk_col)
        fk_map = config.get("fk", {})
        prop_cols = config.get("properties", [])

        rows = load_csv(filename)
        type_records: List[CanonicalRecord] = []

        for row in rows:
            entity_id = row.get(pk_col, "").strip()
            if not entity_id:
                continue

            properties: Dict[str, Any] = {}
            for col in prop_cols:
                properties[col] = parse_cell_value(row.get(col, ""))

            fks: Dict[str, Tuple[str, str, str]] = {}
            for fk_col, (target_label, target_pk) in fk_map.items():
                val = row.get(fk_col, "").strip()
                if val:
                    fks[fk_col] = (target_label, target_pk, val)

            display_name = str(row.get(display_col, entity_id)).strip()
            if len(display_name) > 120:
                display_name = display_name[:117] + "..."

            rec = CanonicalRecord(
                entity_type=label,
                entity_id=entity_id,
                display_name=display_name,
                properties=properties,
                foreign_keys=fks,
                source_file=filename,
                row_hash=compute_row_hash(row),
            )
            type_records.append(rec)

        records_by_type[label] = type_records

    return records_by_type


In [24]:
extract_canonical_records()

{'Department': [CanonicalRecord(entity_type='Department', entity_id='DEPT-ENG', display_name='Engineering', properties={'department_id': 'DEPT-ENG', 'department_name': 'Engineering', 'mission': 'Build and operate reliable product software', 'head_employee_id': 'EMP-0001', 'created_at': '2025-01-01T09:00:00Z'}, foreign_keys={'head_employee_id': ('Employee', 'employee_id', 'EMP-0001')}, source_file='departments.csv', row_hash='64773914a0c7c31f'),
  CanonicalRecord(entity_type='Department', entity_id='DEPT-PROD', display_name='Product', properties={'department_id': 'DEPT-PROD', 'department_name': 'Product', 'mission': 'Define customer problems, roadmap, and product outcomes', 'head_employee_id': 'EMP-0002', 'created_at': '2025-01-01T09:00:00Z'}, foreign_keys={'head_employee_id': ('Employee', 'employee_id', 'EMP-0002')}, source_file='departments.csv', row_hash='d11cbc4fe62921b0'),
  CanonicalRecord(entity_type='Department', entity_id='DEPT-DATA', display_name='Data & AI', properties={'depa